# ContractNLI v2 — 기본 동작 비교와 출력 검증
Windows · PyCharm Jupyter · **Python 3.11.9 · RTX 5070 12GB**.

- Original, Naive, Contextualized, LongLLMLingua, **3개 정답 클래스 모두 유지**.
- 요약 두 조건: 길이 목표 없음, 최대 **2,048 생성 토큰**. 이는 실험 상한이며 Qwen 공식 기본값이라는 뜻이 아닙니다.
- LongLLMLingua: 공식 README 예제 **rate=0.55**, target_token 덮어쓰기 없음. 함수 자체의 rate 기본값은 0.5입니다.
- Qwen3 판단: thinking off, 세 라벨만 허용하는 제약 생성. 추가 학습은 하지 않습니다.
- 실제 길이는 맞추지 않습니다. 이 실행은 기본 동작 비교이고 동일 길이 실험은 별도 분석입니다.

**v1 폴더와 별도의 새 프로젝트 폴더에 이 버전을 설치하세요.** 기존 outputs와 캐시는 보존합니다.
소스/프로토콜이 바뀌었으므로 v1 체크포인트를 v2 추론에 재사용하지 않습니다. 분석은 마지막 audit 셀로 읽기만 합니다.

## 0. PyCharm 준비
새 ZIP을 기존 폴더와 다른 위치에 풉니다. Python 3.11.9 가상환경과 Jupyter 커널을 같은 것으로 선택하세요.
필요한 경우 PyCharm Terminal에서 `python -m pip install jupyter ipykernel`을 실행합니다.
Git 설치는 필요하지 않습니다. Hugging Face 다운로드 캐시는 같은 Windows 계정에서 자동 재사용됩니다.

## 1. 환경 확인 — 재시작 후에도 실행

In [1]:
import os
import sys
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path

print("Python:", sys.version)
print("Kernel executable:", sys.executable)
if sys.version_info[:3] != (3, 11, 9):
    raise RuntimeError(
        "이 노트북은 Python 3.11.9 기준입니다. PyCharm에서 해당 가상환경의 커널을 선택하세요."
    )

# 자동 검색에 실패할 때만 수정하세요. 예: r"C:\Users\YourName\PycharmProjects\JJJ"
PROJECT_ROOT_OVERRIDE = ""

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(
        (p for p in candidates
         if (p / "configs/rtx5070.toml").is_file()
         and (p / "src/contractnli").is_dir()),
        None,
    )

if PROJECT_ROOT is None or not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("PROJECT_ROOT_OVERRIDE에 git clone한 JJJ 폴더 경로를 지정하세요.")

os.chdir(PROJECT_ROOT)

# torch를 처음 import하기 전에 GPU를 한 개만 노출합니다.
if "torch" in sys.modules and os.environ.get("CUDA_VISIBLE_DEVICES") != "0":
    raise RuntimeError("커널을 재시작한 뒤 이 셀부터 다시 실행하세요.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Hugging Face를 import하기 전에 다운로드 대기 시간을 설정합니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
print("Project root:", PROJECT_ROOT)
print("Visible physical GPU:", os.environ["CUDA_VISIBLE_DEVICES"])

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Kernel executable: C:\Users\user\Desktop\JJJ-main1\.venv\Scripts\python.exe
Project root: C:\Users\user\Desktop\JJJ-main1
Visible physical GPU: 0


## 2. 설치 — 새 가상환경에서 한 번
각 설치 셀을 따로 실행하세요. RTX 5070은 torch 2.8.0 / CUDA 12.8 조합을 사용합니다.
설치 후 커널을 재시작하고 **1번 환경 셀 → 3번 GPU 셀**로 이동합니다. Run All로 설치와 추론을 한꺼번에 실행하지 마세요.

In [2]:
%pip install --upgrade pip

  Using cached pip-26.2.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --upgrade torch==2.8.0 --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached torch-2.8.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (29 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
Using cached torch-2.8.0%2Bcu128-cp311-cp311-win_amd64.whl (3461.4 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached filelock-3.32.3-py3-none-any.whl (98 kB)
Using cached fsspec-2026.7.0-py3-none-any.whl (206 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl (15 kB)
Usi

In [4]:
%pip install -e ".[inference,quantization,longllmlingua,rtx5070,analysis]" ipywidgets

Obtaining file:///C:/Users/user/Desktop/JJJ-main1
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached bitsandbytes-0.50.2-py3-none-win_amd64.whl.metadata (11 kB)
  Using cached llmlingua-0.2.2-py3-none-any.whl.metadata (17 kB)
  Using cached sentencepiece-0

## 3. 커널 재시작 후 GPU 확인
재시작 버튼이 보이지 않으면 코드 셀을 클릭하고 **Ctrl+Shift+A → Restart Kernel**을 검색하세요.
재시작 후 1번 환경 셀을 먼저 다시 실행합니다. 아래 두 실제 연산 검사가 모두 PASS여야 합니다.

In [2]:
import gc
import torch
import transformers
import bitsandbytes as bnb

GPU_CHECK_PASSED = False
print("PyTorch:", torch.__version__)
print("PyTorch wheel CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA를 사용할 수 없습니다. 커널, CUDA wheel 설치, NVIDIA 드라이버를 확인하세요.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("커널을 재시작하고 1번 셀에서 GPU 0 노출 설정을 적용하세요.")
if torch.__version__.split("+")[0] != "2.8.0" or torch.version.cuda != "12.8":
    raise RuntimeError("이 실험 프로필은 torch 2.8.0/CUDA 12.8입니다. 설치 셀 실행 후 커널을 재시작하세요.")
if bnb.__version__ != "0.48.1":
    raise RuntimeError("rtx5070 extra로 bitsandbytes 0.48.1을 설치하고 커널을 재시작하세요.")

properties = torch.cuda.get_device_properties(0)
capability = torch.cuda.get_device_capability(0)
architectures = torch.cuda.get_arch_list()
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print("GPU:", properties.name)
print("Compute capability:", capability)
print("Wheel architectures:", architectures)
print(f"VRAM total: {properties.total_memory / 1024**3:.2f} GiB")
print(f"VRAM currently free: {free_bytes / 1024**3:.2f} GiB")
if capability >= (12, 0) and "sm_120" not in architectures:
    raise RuntimeError("현재 PyTorch wheel에 Blackwell sm_120 지원이 없습니다. CUDA 12.8 설치 셀을 확인하세요.")

# CUDA available=True만으로 실제 커널 호환성을 확인할 수는 없습니다.
with torch.inference_mode():
    matrix = torch.randn(64, 64, device="cuda:0", dtype=torch.float16)
    product = matrix @ matrix.T
    torch.cuda.synchronize()
    if not torch.isfinite(product).all().item():
        raise RuntimeError("CUDA 연산 결과가 유효하지 않습니다.")
print("CUDA tensor test: PASS")

# 대형 모델을 받기 전에 Windows/Blackwell NF4 커널을 직접 확인합니다.
with torch.inference_mode():
    layer = bnb.nn.Linear4bit(
        64, 32, bias=False, compute_dtype=torch.float16,
        quant_type="nf4", compress_statistics=True,
    ).to("cuda:0")
    probe_input = torch.randn(2, 64, device="cuda:0", dtype=torch.float16)
    probe_output = layer(probe_input)
    torch.cuda.synchronize()
    if probe_output.shape != (2, 32) or not torch.isfinite(probe_output).all().item():
        raise RuntimeError("NF4 4bit 연산 결과가 유효하지 않습니다.")
print("bitsandbytes NF4 test: PASS")

del matrix, product, layer, probe_input, probe_output
gc.collect()
torch.cuda.empty_cache()
GPU_CHECK_PASSED = True
print("GPU compatibility checks: PASS. 다음 설정 셀로 진행하세요.")

PyTorch: 2.8.0+cu128
PyTorch wheel CUDA: 12.8
Transformers: 4.51.3
bitsandbytes: 0.48.1
CUDA available: True
GPU: NVIDIA GeForce RTX 5070
Compute capability: (12, 0)
Wheel architectures: ['sm_61', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
VRAM total: 11.94 GiB
VRAM currently free: 10.78 GiB
CUDA tensor test: PASS
bitsandbytes NF4 test: PASS
GPU compatibility checks: PASS. 다음 설정 셀로 진행하세요.


## 4. 프로토콜·모델 revision 고정
처음에는 메타데이터만 읽어 모델 세 개의 실제 commit을 고정합니다. 가중치를 로딩하지 않습니다.
`configs/rtx5070_v2.frozen.json`은 이후 실행에도 재사용합니다. 모델이나 설정을 바꾸려면 새 config와 출력 폴더를 만드세요.

In [3]:
if not globals().get("GPU_CHECK_PASSED", False):
    raise RuntimeError("GPU 실제 연산 검사부터 통과하세요.")
import json
from dataclasses import replace
import contractnli
from contractnli.config import load_config
from contractnli.reproducibility import freeze_config
from contractnli.experiment import Experiment, inspect
from contractnli.data import load_dataset
from contractnli.download import download
from contractnli.preflight import preflight

expected_package = PROJECT_ROOT / "src" / "contractnli" / "__init__.py"
if Path(contractnli.__file__).resolve() != expected_package.resolve():
    raise RuntimeError("다른 프로젝트의 패키지가 로딩됐습니다. 이 폴더에서 설치한 뒤 커널을 재시작하세요.")
BASE_CONFIG = freeze_config(load_config(PROJECT_ROOT / "configs" / "rtx5070_v2.toml"),
                            PROJECT_ROOT / "configs" / "rtx5070_v2.frozen.json")
print(json.dumps(BASE_CONFIG.to_dict(), indent=2, ensure_ascii=False))

{
  "dataset": "data/contract-nli/test.json",
  "expected_dataset_sha256": "460267b56052a2dc5aead98eb35eadef9e6734d5723d37b4a9790e410f812387",
  "output_dir": "outputs/rtx5070_native_v2",
  "conditions": [
    "original",
    "naive",
    "contextualized",
    "longllmlingua"
  ],
  "retention_ratio": 0.55,
  "judge_max_new_tokens": 16,
  "seed": 42,
  "max_documents": null,
  "judge_decoding": "constrained",
  "budget_policy": "target",
  "budget_attempt_limit": 6,
  "warmup_calls": 1,
  "shuffle_judgments": true,
  "require_pinned_revisions": true,
  "require_freeze": true,
  "summary_length_policy": "fixed",
  "summary_max_new_tokens": 2048,
  "compressor_budget_mode": "rate",
  "summarizer": {
    "name": "Qwen/Qwen2.5-7B-Instruct",
    "revision": "a09a35458c702b33eeacc393d103063234e8bc28",
    "device": "cuda",
    "dtype": "float16",
    "quantization": "4bit",
    "max_input_tokens": 32768
  },
  "judge": {
    "name": "Qwen/Qwen3-8B",
    "revision": "b968826d9c46dd6066d109eab

## 5. 데이터 확인
전체 test는 123문서, 2,091문항입니다. 실행 검사는 dev 1문서를 사용합니다.
이전에 test 결과를 확인했으므로 이후 조정은 탐색적 실험입니다. 프롬프트·모델 설정의 추가 튜닝은 dev에서 수행하세요.

In [4]:
DATA_PATH = PROJECT_ROOT / BASE_CONFIG.dataset
DEV_PATH = DATA_PATH.with_name("dev.json")
if not DATA_PATH.is_file() or not DEV_PATH.is_file():
    download(DATA_PATH.parent)
from contractnli.data import load_for_config
dataset = load_for_config(BASE_CONFIG)
inventory = inspect(BASE_CONFIG, dataset)
print(json.dumps(inventory, indent=2, ensure_ascii=False))
if inventory["documents"] != 123 or inventory["evaluated_pairs"] != 2091:
    raise RuntimeError("공식 전체 test 데이터가 아닙니다.")
SMOKE_CONFIG = replace(BASE_CONFIG, dataset=str(DEV_PATH), max_documents=1,
                       expected_dataset_sha256="310af7d661d2ab50ee3700169cef524c75f39fb296bbf5a515c229eb0f42e68e",
                       output_dir="outputs/rtx5070_native_v2_dev_smoke")

{
  "documents": 123,
  "evaluated_pairs": 2091,
  "gold_label_counts": {
    "NotMentioned": 903,
    "Entailment": 968,
    "Contradiction": 220
  },
  "gold_evidence_available_pairs": 1188,
  "not_mentioned_pairs": 903,
  "conditions": [
    "original",
    "naive",
    "contextualized",
    "longllmlingua"
  ],
  "target_token_retention": null,
  "summary_length_policy": "fixed",
  "summary_max_new_tokens": 2048,
  "compressor_budget_mode": "rate",
  "compressor_rate": 0.55,
  "compressor_context_target_ratio": null,
  "judge_decoding": "constrained",
  "budget_policy": "target",
  "summary_requests": 246,
  "question_aware_compression_requests": 2091,
  "judgment_requests": 8364,
  "dataset_sha256": "460267b56052a2dc5aead98eb35eadef9e6734d5723d37b4a9790e410f812387",
  "partial_dataset": false
}


## 6. 입력 길이 검사
원문을 자동으로 자르지 않습니다. 요약은 입력 길이 + 2,048 출력 토큰을 확인합니다.
이 단계는 토크나이저만 사용하므로 VRAM 적재 여부는 다음 단계에서 확인합니다.
LongLLMLingua prefix 호환 처리는 코드에 통합되어 별도 패치 셀이 필요하지 않습니다.

In [5]:
length_check = preflight(BASE_CONFIG)
print(json.dumps(length_check, indent=2, ensure_ascii=False))
if not length_check["ok"]:
    raise RuntimeError("입력 길이 한도를 초과했습니다. 원문을 임의로 자르지 마세요.")

{
  "ok": true,
  "maxima": {
    "summary_prompt_plus_output_tokens": 10056,
    "original_judge_prompt_plus_output_tokens": 8074,
    "longllmlingua_chunks_per_document": 25
  },
  "context_overflows": [],
  "constrained_label_token_lengths": {
    "Entailment": 3,
    "Contradiction": 3,
    "NotMentioned": 4
  },
  "note": "Tokenizer-only check. GPU availability, 4bit loading and peak VRAM require an actual smoke run."
}


## 7. dev 한 문서 실행 검사
68개 판단을 완료할 때까지 기다리세요. 첫 실행은 모델 다운로드도 필요합니다.
모델마다 고정된 합성 예제로 1회 워밍업하며, 워밍업은 점수와 처리시간 집계에서 제외됩니다.
`Loading`, `summary`, `compression`, `judgment` 로그를 구분해서 보세요.
**Smoke run completed: True**와 Invalid=0을 확인한 뒤 8번으로 이동합니다.

In [7]:
smoke_report = None
print("dev 한 문서 실행 검사를 시작합니다. 완료될 때까지 중지하지 마세요.", flush=True)
smoke_report = Experiment(SMOKE_CONFIG).run("all")
for condition, m in smoke_report["metrics"].items():
    print(condition, "Invalid:", m["invalid_outputs"], "판단 길이 제한 도달:", m["judgment_length_capped_calls"])
    if m["invalid_outputs"] or m["judgment_length_capped_calls"]:
        raise RuntimeError("출력 프로토콜 검증 실패. 전체 실행을 시작하지 마세요.")
print("Smoke run completed:", smoke_report["complete"])
print("실행 검사 결과는 dev 부분집합으로 test 성능이 아닙니다.")

dev 한 문서 실행 검사를 시작합니다. 완료될 때까지 중지하지 마세요.
original Invalid: 0 판단 길이 제한 도달: 0
naive Invalid: 0 판단 길이 제한 도달: 0
contextualized Invalid: 0 판단 길이 제한 도달: 0
longllmlingua Invalid: 0 판단 길이 제한 도달: 0
Smoke run completed: True
실행 검사 결과는 dev 부분집합으로 test 성능이 아닙니다.


## 8. 전체 메인 실험
7번이 끝나면 아래 변수만 True로 바꾸고 이 셀을 실행합니다.
단일 GPU에서 요약 → 압축 → 판단 모델을 순서대로 올리고 해제합니다.
판단은 특정 조건이 항상 먼저 실행되지 않도록 seed=42의 고정 해시 순서로 섞습니다.
일반 중단 후 같은 셀을 재실행하면 완료된 항목을 재사용합니다. 기존 v1 결과는 읽거나 덮어쓰지 않습니다.

In [8]:
RUN_FULL_EXPERIMENT = True
if RUN_FULL_EXPERIMENT:
    if not isinstance(globals().get("smoke_report"), dict) or not smoke_report.get("complete"):
        raise RuntimeError("7번 dev 실행 검사가 완료되어야 합니다.")
    if any(m["invalid_outputs"] for m in smoke_report["metrics"].values()):
        raise RuntimeError("dev 출력 검증을 통과하지 못했습니다.")
    full_report = Experiment(BASE_CONFIG).run("all")
    print("전체 실험 완료:", full_report["complete"])
    print("결과 폴더:", (PROJECT_ROOT / BASE_CONFIG.output_dir).resolve())
else:
    print("전체 실행은 꺼져 있습니다. dev 검사가 끝난 뒤 True로 바꾸세요.")

Loading summarizer: Qwen/Qwen2.5-7B-Instruct (download/loading time is recorded separately)


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

summary 1/156: naive/83 [11.58s]
summary 2/156: contextualized/83 [42.40s]
summary 3/156: naive/84 [11.98s]
summary 4/156: contextualized/84 [54.20s]
summary 5/156: naive/85 [8.42s]
summary 6/156: contextualized/85 [34.71s]
summary 7/156: naive/90 [12.07s]
summary 8/156: contextualized/90 [29.56s]
summary 9/156: naive/97 [16.24s]
summary 10/156: contextualized/97 [23.95s]
summary 11/156: naive/99 [5.95s]
summary 12/156: contextualized/99 [31.31s]
summary 13/156: naive/117 [7.35s]
summary 14/156: contextualized/117 [50.11s]
summary 15/156: naive/127 [15.56s]
summary 16/156: contextualized/127 [53.77s]
summary 17/156: naive/129 [16.22s]
summary 18/156: contextualized/129 [39.11s]
summary 19/156: naive/138 [10.98s]
summary 20/156: contextualized/138 [50.34s]
summary 21/156: naive/157 [9.09s]
summary 22/156: contextualized/157 [29.65s]
summary 23/156: naive/165 [15.44s]
summary 24/156: contextualized/165 [70.14s]
summary 25/156: naive/181 [13.49s]
summary 26/156: contextualized/181 [72.11s

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

compression 1/2091: 1/nda-11 [5.90s]
compression 2/2091: 1/nda-16 [6.00s]
compression 3/2091: 1/nda-15 [5.94s]
compression 4/2091: 1/nda-10 [5.90s]
compression 5/2091: 1/nda-2 [5.90s]
compression 6/2091: 1/nda-1 [5.90s]
compression 7/2091: 1/nda-19 [5.92s]
compression 8/2091: 1/nda-12 [5.88s]
compression 9/2091: 1/nda-20 [5.96s]
compression 10/2091: 1/nda-3 [5.96s]
compression 11/2091: 1/nda-18 [5.94s]
compression 12/2091: 1/nda-7 [5.91s]
compression 13/2091: 1/nda-17 [5.95s]
compression 14/2091: 1/nda-8 [6.04s]
compression 15/2091: 1/nda-13 [5.96s]
compression 16/2091: 1/nda-5 [5.92s]
compression 17/2091: 1/nda-4 [5.93s]
compression 18/2091: 2/nda-11 [5.93s]
compression 19/2091: 2/nda-16 [5.55s]
compression 20/2091: 2/nda-15 [5.56s]
compression 21/2091: 2/nda-10 [5.54s]
compression 22/2091: 2/nda-2 [6.25s]
compression 23/2091: 2/nda-1 [6.22s]
compression 24/2091: 2/nda-19 [5.51s]
compression 25/2091: 2/nda-12 [5.53s]
compression 26/2091: 2/nda-20 [5.55s]
compression 27/2091: 2/nda-3 [

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

judgment 1/8364: original/596/nda-2 [1.20s]
judgment 2/8364: original/83/nda-19 [0.60s]
judgment 3/8364: longllmlingua/18/nda-18 [0.47s]
judgment 4/8364: naive/2/nda-5 [0.39s]
judgment 5/8364: original/42/nda-16 [0.78s]
judgment 6/8364: original/542/nda-13 [1.37s]
judgment 7/8364: naive/50/nda-7 [0.42s]
judgment 8/8364: contextualized/354/nda-10 [0.70s]
judgment 9/8364: naive/412/nda-4 [0.31s]
judgment 10/8364: longllmlingua/600/nda-12 [0.49s]
judgment 11/8364: contextualized/181/nda-15 [0.67s]
judgment 12/8364: longllmlingua/596/nda-11 [0.77s]
judgment 13/8364: contextualized/49/nda-5 [0.51s]
judgment 14/8364: longllmlingua/85/nda-12 [0.41s]
judgment 15/8364: original/52/nda-17 [1.07s]
judgment 16/8364: longllmlingua/18/nda-17 [0.48s]
judgment 17/8364: longllmlingua/59/nda-7 [0.60s]
judgment 18/8364: longllmlingua/455/nda-1 [0.37s]
judgment 19/8364: naive/47/nda-13 [0.39s]
judgment 20/8364: naive/497/nda-7 [0.43s]
judgment 21/8364: naive/395/nda-5 [0.43s]
judgment 22/8364: longllmling

In [5]:
import json
from dataclasses import replace
from pathlib import Path
from contractnli.experiment import Experiment

DEV_JUDGE_CONFIG = replace(
    BASE_CONFIG,
    dataset=str(
        (PROJECT_ROOT / BASE_CONFIG.dataset).with_name("dev.json")
    ),
    expected_dataset_sha256=(
        "310af7d661d2ab50ee3700169cef524c75f39"
        "fb296bbf5a515c229eb0f42e68e"
    ),
    conditions=("original",),
    max_documents=None,
    output_dir="outputs/rtx5070_v2_judge_dev",
)

print("dev Original 판단 검증을 시작합니다.", flush=True)

dev_judge_report = Experiment(DEV_JUDGE_CONFIG).run("all")
m = dev_judge_report["metrics"]["original"]

print("\n검증 완료:", dev_judge_report["complete"])
print("문항 수:", m["n"])
print("Accuracy:", round(m["accuracy"], 4))
print("Macro-F1:", round(m["macro_f1"], 4))
print("Invalid:", m["invalid_outputs"])

print("\n클래스별 성능:")
print(json.dumps(m["per_label"], indent=2, ensure_ascii=False))

print("\n혼동행렬 — 행: 정답, 열: 예측")
print(json.dumps(m["confusion"], indent=2, ensure_ascii=False))

print("\n결과 폴더:")
print((PROJECT_ROOT / DEV_JUDGE_CONFIG.output_dir).resolve())

dev Original 판단 검증을 시작합니다.
Loading judge: Qwen/Qwen3-8B (download/loading time is recorded separately)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True, 'temperature': 0.6, 'top_k': 20, 'top_p': 0.95, 'bos_token_id': 151643}. If this is not desired, please set these values explicitly.


judgment 1/1037: original/70/nda-18 [0.75s]
judgment 2/1037: original/15/nda-8 [0.64s]
judgment 3/1037: original/72/nda-10 [1.34s]
judgment 4/1037: original/69/nda-16 [1.38s]
judgment 5/1037: original/507/nda-11 [0.75s]
judgment 6/1037: original/486/nda-10 [0.77s]
judgment 7/1037: original/37/nda-2 [1.48s]
judgment 8/1037: original/478/nda-20 [1.13s]
judgment 9/1037: original/381/nda-13 [1.24s]
judgment 10/1037: original/19/nda-8 [0.83s]
judgment 11/1037: original/37/nda-1 [1.48s]
judgment 12/1037: original/15/nda-15 [0.55s]
judgment 13/1037: original/411/nda-19 [0.92s]
judgment 14/1037: original/14/nda-5 [0.71s]
judgment 15/1037: original/63/nda-12 [0.96s]
judgment 16/1037: original/69/nda-17 [1.40s]
judgment 17/1037: original/582/nda-11 [1.25s]
judgment 18/1037: original/610/nda-19 [1.37s]
judgment 19/1037: original/81/nda-3 [0.65s]
judgment 20/1037: original/71/nda-8 [0.51s]
judgment 21/1037: original/39/nda-3 [0.69s]
judgment 22/1037: original/29/nda-10 [0.71s]
judgment 23/1037: or

In [6]:
import json
from collections import Counter

path = PROJECT_ROOT / "outputs/rtx5070_v2_judge_dev/predictions.jsonl"
rows = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
contradictions = [r for r in rows if r["gold"] == "Contradiction"]

for hid in sorted({r["hypothesis_id"] for r in contradictions}):
    group = [r for r in contradictions if r["hypothesis_id"] == hid]

    print("\n가설:", hid, group[0]["hypothesis"])
    print("예측 분포:", dict(Counter(r["prediction"] for r in group)))

    for prediction in ("NotMentioned", "Entailment"):
        example = next(
            (r for r in group if r["prediction"] == prediction), None
        )
        if example:
            print("오답 사례:", example["example_id"])
            print("예측:", prediction)
            print("공식 정답 근거:", example["gold_evidence"])


가설: nda-1 All Confidential Information shall be expressly identified by the Disclosing Party.
예측 분포: {'NotMentioned': 9, 'Entailment': 1}
오답 사례: 37/nda-1
예측: NotMentioned
공식 정답 근거: ['“Confidential Information” means:', '(b) other Information provided by the Disclosing Party pursuant to [or in anticipation of] this Agreement that is clearly designated as being confidential or equivalent (whether or not it is so marked) or that ought reasonably to be considered to be confidential which comes (or has come) to the Receiving Party’s attention or into the Receiving Party’s possession in connection with the Permitted Purpose;']
오답 사례: 478/nda-1
예측: Entailment
공식 정답 근거: ['Notwithstanding the foregoing, the components of the System are hereby deemed to be Proprietary Information and as such are protected in accordance with the terms of this Agreement.']

가설: nda-16 Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.
예측 분포: {'Entailment': 2}


## 9. 성능·형식·시간 함께 확인
시간 단위는 분입니다. 총시간은 요약/압축 + 판단이며, 모델 준비·워밍업·체크포인트 I/O는 제외합니다.
모델 준비와 실행 전체 경과시간은 run.json에 별도로 기록합니다. 저장된 시간을 재사용한 실행은 새 벤치마크가 아닙니다.

In [9]:
from IPython.display import Markdown, display
result_config = BASE_CONFIG if globals().get("RUN_FULL_EXPERIMENT", False) else SMOKE_CONFIG
result_directory = PROJECT_ROOT / result_config.output_dir
metrics_path = result_directory / "metrics.json"
if not metrics_path.exists():
    print("아직 전체 예측이 완료되지 않았습니다:", result_directory)
else:
    report = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("부분집합:", report["dataset"]["partial_dataset"], "문항 수:", report["dataset"]["evaluated_pairs"])
    table = ["| 조건 | Accuracy | Macro-F1 | Invalid | 실제 보존율 | 원문 정답 유지율 |",
             "|---|---:|---:|---:|---:|---:|"]
    timing = ["| 조건 | 요약·압축(분) | 판단(분) | 합계(분) | 문항당(초) | 판단 p95(초) |",
              "|---|---:|---:|---:|---:|---:|"]
    for condition, m in report["metrics"].items():
        retention = m.get("correct_decision_retention")
        keep = "—" if retention is None else f"{retention:.2%}"
        table.append(f"| {condition} | {m['accuracy']:.2%} | {m['macro_f1']:.4f} | {m['invalid_outputs']} | {m['mean_actual_token_retention']:.2%} | {keep} |")
        timing.append(f"| {condition} | {m['compression_seconds']/60:.2f} | {m['judgment_seconds']/60:.2f} | {m['total_inference_seconds']/60:.2f} | {m['amortized_seconds_per_pair']:.3f} | {m['judgment_p95_seconds']:.3f} |")
        print(condition, "요약 생성 상한 도달:", m["length_capped_summary_calls"], "판단 출력 상한 도달:", m["judgment_length_capped_calls"])
    display(Markdown("\n".join(table)))
    display(Markdown("\n".join(timing)))
    print("JSON:", metrics_path)
    print("Excel용 CSV:", result_directory / "metrics.csv")

부분집합: False 문항 수: 2091
original 요약 생성 상한 도달: 0 판단 출력 상한 도달: 0
naive 요약 생성 상한 도달: 0 판단 출력 상한 도달: 0
contextualized 요약 생성 상한 도달: 4 판단 출력 상한 도달: 0
longllmlingua 요약 생성 상한 도달: 0 판단 출력 상한 도달: 0


| 조건 | Accuracy | Macro-F1 | Invalid | 실제 보존율 | 원문 정답 유지율 |
|---|---:|---:|---:|---:|---:|
| original | 66.24% | 0.5270 | 0 | 100.00% | — |
| naive | 55.52% | 0.4272 | 0 | 16.92% | 72.13% |
| contextualized | 64.42% | 0.5488 | 0 | 54.75% | 87.58% |
| longllmlingua | 60.02% | 0.4437 | 0 | 55.22% | 79.86% |

| 조건 | 요약·압축(분) | 판단(분) | 합계(분) | 문항당(초) | 판단 p95(초) |
|---|---:|---:|---:|---:|---:|
| original | 0.00 | 33.29 | 33.29 | 0.955 | 1.835 |
| naive | 28.57 | 14.15 | 42.72 | 1.226 | 0.475 |
| contextualized | 89.72 | 19.40 | 109.12 | 3.131 | 0.750 |
| longllmlingua | 251.01 | 22.53 | 273.54 | 7.849 | 1.061 |

JSON: C:\Users\user\Desktop\JJJ-main1\outputs\rtx5070_native_v2\metrics.json
Excel용 CSV: C:\Users\user\Desktop\JJJ-main1\outputs\rtx5070_native_v2\metrics.csv


## 10. 선택: 기존 또는 새 결과의 읽기 전용 분석
모델을 올리지 않습니다. Accuracy/Macro-F1 재계산, 엄격한 파싱/첫 줄 파싱 비교, 클래스별·가설별 지표,
원문 정답 보존, 문서 단위 bootstrap 20,000회를 별도 audit 폴더에 저장합니다.

길이 겹침 진단은 같은 문항에서 보존율 차이가 2%p 이내인 부분집합만 보여줍니다. 선택된 표본이므로
동일 길이 전체 실험으로 해석하면 안 됩니다. 정확히 같은 토큰 수에서 비용/시간을 비교하는 벤치마크는 별도 프로토콜입니다.

In [10]:
RUN_AUDIT = False
# 기존 v1 분석 시 이 경로만 기존 결과 폴더로 바꾸세요. 원본 파일은 변경하지 않습니다.
AUDIT_SOURCE = PROJECT_ROOT / BASE_CONFIG.output_dir
if RUN_AUDIT:
    from contractnli.audit import save_audit
    audit = save_audit(AUDIT_SOURCE / "predictions.jsonl", AUDIT_SOURCE / "metrics.json",
                       PROJECT_ROOT / "outputs" / "native_v2_audit", bootstrap_samples=20000)
    print("분석 완료:", PROJECT_ROOT / "outputs" / "native_v2_audit")

## 해석과 재현 범위
- 메인 결과는 실제 길이가 다른 기본 동작 비교입니다. 길이 효과와 방법 효과를 분리해 주장하지 않습니다.
- Original의 오답 유지와 정답 유지는 다릅니다. 정답 유지율·정답→오답·오답→정답을 함께 보세요.
- NotMentioned는 정답 라벨로 유지하되, 없는 gold evidence를 근거 보존 0점으로 계산하지 않습니다.
- 생성 상한에 도달한 요약은 잘린 문장인지 확인합니다. 자동 연장·절단·정답 기반 재시도는 하지 않습니다.
- Constrained decoding은 출력 형식을 보장합니다. 정답률·편향 제거·비트 단위 완전 재현을 보장하는 것은 아닙니다.
- 제약 생성과 길이 설정이 바뀌었으므로 v1과 v2는 별도 프로토콜입니다. 기존 결과를 새 결과로 덮어쓰지 않습니다.
- 모델 변경이나 추가 학습을 진행하면 별도 실험으로 기록하고 train/dev만 학습·튜닝에 사용하세요.